# Monta Meu Pedido · Fine-tuning do BERTimbau

Treina o classificador multi-label que diz quais destas 7 tags um prato tem:
`vegano`, `vegetariano`, `sem_lactose`, `sem_gluten`, `fit`, `picante`, `doce`.

**O que este notebook faz**
1. Confere a GPU
2. Baixa o código e os dados do GitHub
3. Faz login no Hugging Face
4. Treina, escolhe os limiares na validação, avalia **uma vez** no teste e publica o modelo
5. Testa o modelo publicado

A lógica de treino fica em `src/mmp/models/bertimbau.py`, no repositório. O notebook só orquestra:
assim o mesmo código é testado, versionado e reutilizado.

> Antes de rodar: **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU**.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Código e dados

Clona o repositório (que já contém os splits de treino, validação e teste), instala as dependências e deixa o pacote `mmp` visível para o Python.

In [ ]:
REPO = "https://github.com/kesiasilv/monta-meu-pedido.git"
MODELO_HF = "KesiaSilva/bertimbau-tags-pratos"

!git clone -q {REPO}
%cd monta-meu-pedido
!pip install -q pydantic-settings iterative-stratification "transformers>=4.46" "datasets>=3.0" accelerate

# O Colab só enxerga pacotes instalados com "pip install -e" depois de reiniciar a sessão.
# Adicionar a pasta src ao caminho resolve sem reiniciar.
import sys
sys.path.insert(0, "/content/monta-meu-pedido/src")

## 3. Login no Hugging Face

Abra o link que aparecer, digite o código e autorize com acesso de escrita. Se aparecer uma caixa de texto, cole o seu token de tipo **Write**.

In [ ]:
from huggingface_hub import login

login()

## 4. Treinar e publicar

- Até 8 épocas, com early stopping pelo F1 macro da validação
- Limiar de cada tag escolhido na validação
- Avaliação única no teste
- Publica o modelo, o tokenizer e o model card em `MODELO_HF`

Leva poucos minutos na T4.

In [ ]:
import logging

from mmp.models.bertimbau import treinar

logging.basicConfig(level=logging.INFO)
resultado = treinar(push_para=MODELO_HF)

In [ ]:
print("Limiares por tag:", resultado["limiares"])
for conjunto in ("validacao", "teste"):
    m = resultado[conjunto]
    print(f"\n{conjunto.upper()}: F1 micro {m['f1_micro']:.3f} | F1 macro {m['f1_macro']:.3f}")
    for tag, t in m["por_tag"].items():
        print(f"  {tag:<12} F1 {t['f1']:.3f}  (suporte {t['suporte']})")

## 5. Testar o modelo publicado

Baixa o modelo do Hub, do mesmo jeito que o workflow vai fazer.

In [ ]:
from mmp.models.bertimbau import BertimbauClassificador

clf = BertimbauClassificador(MODELO_HF)
exemplos = [
    "Poke de tofu. Tofu grelhado com arroz e legumes. Ingredientes: tofu, arroz, pepino, gergelim",
    "Lasanha à bolonhesa. Massa com molho de carne e queijo. Ingredientes: massa de trigo, carne moída, queijo, molho de tomate",
    "Brigadeiro gourmet. Doce de chocolate. Ingredientes: leite condensado, chocolate, manteiga",
]
for texto, tags in zip(exemplos, clf.prever(exemplos)):
    print(f"{texto.split('.')[0]:<22} -> {sorted(tags)}")

## 6. Guardar as métricas

Baixe o arquivo abaixo e coloque em `artifacts/bertimbau/metricas.json` no projeto, para o relatório.

In [ ]:
from google.colab import files

files.download("artifacts/bertimbau/metricas.json")